In [8]:

import pandas as pd
df = pd.read_csv("../data/train.csv")
df.head()
df.info()
df.isnull().sum()
(df.isnull().sum() / len(df)) * 100
df["Embarked"].value_counts()
df["Embarked"] = df["Embarked"].fillna("S")
df["Survived"].value_counts()
pd.crosstab(df["Sex"], df["Survived"])
df.groupby("Survived")["Age"].mean()
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 10, 20, 30, 40, 50, 60, 100],
    labels=["0-10", "10-20", "20-30", "30-40", "40-50", "50-60", "60+"]
)
df[["Age", "AgeGroup"]].head(10)
df.groupby("AgeGroup", observed=True)["Survived"].mean()
df.groupby(df["Cabin"].isnull())["Survived"].mean()
df.groupby("Pclass")["Survived"].mean()
pd.crosstab(df["Pclass"], df["Cabin"].isnull())
df.groupby("Survived")["Fare"].mean()
df.groupby(["Pclass", "Sex"])["Age"].mean()
df.groupby(["Pclass", "Sex"])["Age"].median()
df["Age"]=df["Age"].fillna(df.groupby(["Pclass","Sex"])["Age"].transform("median"))
df["Age"].isnull().sum()
df["Age"].head(10)
df.isnull().sum()
df = df.drop(columns=["AgeGroup"])
df["Embarked"] = df["Embarked"].fillna("S")

df["Age"] = df["Age"].fillna(
    df.groupby(["Pclass", "Sex"])["Age"].transform("median")
)
df.isnull().sum()
df["CabinKnown"] = df["Cabin"].notnull().astype(int)
df["Cabin"].notnull().astype(int)
df["Cabin"].notnull().astype(int)
df["CabinKnown"].value_counts()
df["Sex_Pclass"] = (
    df["Sex"] + "_" + df["Pclass"].astype(str)
)
df[["Sex", "Pclass", "Sex_Pclass"]].head()
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df[["SibSp", "Parch", "FamilySize"]].head(10)
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")
df["Title"].value_counts()
df["TitleGrouped"] = df["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

rare_titles = [
    "Dr", "Rev", "Major", "Col", "Don",
    "Lady", "Sir", "Capt", "the Countess", "Jonkheer"
]

df["TitleGrouped"] = df["TitleGrouped"].replace(
    rare_titles,
    "Rare"
)

df["TitleGrouped"].value_counts()
df.columns
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "CabinKnown",
    "FamilySize", "TitleGrouped"
]

X = df[features]
y = df["Survived"]
X.head()
y.head()
X.dtypes
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "CabinKnown"
]

categorical_features = [
    "Sex",
    "Embarked", "TitleGrouped"
]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)
#X_processed=preprocessor.fit_transform(X)
#X_processed.shape
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val= train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
X_train.shape, X_val.shape, y_train.shape, y_val.shape
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
model=Pipeline(steps=[("preprocessor", preprocessor),("classifier", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
y_pred=model.predict(X_val)
y_pred[:10]
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_val, y_pred)

accuracy
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, y_pred)

cm
from sklearn.metrics import classification_report
print(classification_report(y_val,y_pred))
results=X_val.copy()
results["actual"]=y_val
results["predicted"]=y_pred
results.head()
mistakes=results[results["actual"]!=results["predicted"]]
mistakes
mistakes.shape
false_negatives = results[
    (results["actual"] == 1) &
    (results["predicted"] == 0)
]

false_positives = results[
    (results["actual"] == 0) &
    (results["predicted"] == 1)
]
false_negatives.shape, false_positives.shape
false_negatives.sort_values(
    ["Pclass", "Sex", "Age"]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
model_interaction = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)
model_interaction.fit(X_train, y_train)
y_pred_interaction = model_interaction.predict(X_val)
accuracy_interaction = accuracy_score(
    y_val,
    y_pred_interaction
)

accuracy_interaction
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)
cm_interaction = confusion_matrix(
    y_val,
    y_pred_interaction
)

cm_interaction
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)
model_family = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)
model_family.fit(X_train, y_train)
y_pred_family = model_family.predict(X_val)
accuracy_family = accuracy_score(y_val, y_pred_family)

accuracy_family
from sklearn.metrics import classification_report

print(classification_report(y_val, y_pred_family))

categorical_features=["Sex", "Embarked", "TitleGrouped"]
accuracy_score(y_val, y_pred_family)
model_title = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)
model_title.fit(X_train, y_train)
y_pred_title = model_title.predict(X_val)
accuracy_title = accuracy_score(y_val, y_pred_title)

accuracy_title


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
              precision    recall  f1-score   support

           0       0.86      0.88      0.87       110
           1       0.80      0.77      0.79        69

    accuracy                           0.84       179
   macro avg       0.83      0.82      0.

0.8379888268156425